# Лекция 5. Ускорение и масштабирование обучения — код

Код из лекции: где теряется время при обучении, `DataLoader` с рабочими процессами, профилировщик, память при обучении, форматы float16 и bfloat16, смешанная точность и GradScaler, gradient checkpointing, большие батчи, разогрев, накопление и ограничение градиента, параллелизм по данным (настоящий запуск DDP на двух процессах), тензорный параллелизм, ZeRO, чекпоинты, шум и mixup.

Ячейки выполняются **по порядку**: каждая следующая использует переменные из предыдущих. Проще всего — «Среда выполнения → Выполнить все». Весь ноутбук выполняется за несколько минут.

Код работает и на процессоре, и на видеокарте: в первой ячейке выбирается устройство `device`. Пояснения ниже написаны по запуску **на процессоре**. Если включить GPU («Среда выполнения → Сменить среду выполнения»), смешанная точность будет во float16 с GradScaler и обычно ускорит обучение, а не замедлит его. Время в любом случае будет другим, а последние знаки чисел могут немного отличаться.

Код, которому нужна видеокарта, несколько машин или Lightning, показан как текст — его запускать не нужно. На своём компьютере понадобятся `pip install torch scikit-learn`.

## Глава 1. Где теряется время

### 1.2. Замеряем: ожидание данных и вычисления

Данные — рукописные цифры 8×8 (1347 примеров для обучения, 450 для валидации), сеть `DeepMLP` из лекции 4: 12 pre-norm блоков `x + F(LayerNorm(x))` с расширением в 4 раза. Здесь же функции `n_params` (число параметров) и `evaluate` (точность на валидации) — они нужны до конца ноутбука.

In [ ]:
import time                                           # замеры времени
import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# видеокарта, если она есть, иначе процессор
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device)

# данные — те же рукописные цифры 8×8, что в лекции 4
X, y = load_digits(return_X_y=True)
# 75% примеров — на обучение, 25% — на валидацию; stratify=y сохраняет доли классов в обеих частях
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)                # среднее и разброс признаков — только по обучающей части
X_train = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
X_val = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
y_train, y_val = torch.tensor(y_train), torch.tensor(y_val)

class Block(nn.Module):
    """Pre-norm остаточный блок из лекции 4: x + F(LayerNorm(x)), ветка расширяет ширину в 4 раза."""
    def __init__(self, width, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(width)               # нормализация входа ветки
        # ветка F: расширение width → 4·width, активация, dropout, сжатие обратно до width
        self.f = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Dropout(dropout),
                               nn.Linear(4 * width, width))

    def forward(self, x):
        return x + self.f(self.norm(x))               # обходной путь x плюс выход ветки

class DeepMLP(nn.Module):
    """Сеть из лекции 4 (раздел 5.3), без DropPath: stem → блоки → LayerNorm → head."""
    def __init__(self, d_in=64, n_classes=10, width=64, n_blocks=12, dropout=0.1):
        super().__init__()
        self.stem = nn.Linear(d_in, width)            # вход: 64 пикселя → width признаков
        self.blocks = nn.Sequential(*[Block(width, dropout) for _ in range(n_blocks)])
        self.norm = nn.LayerNorm(width)               # финальная нормализация
        self.head = nn.Linear(width, n_classes)       # выход: 10 логитов, по одному на цифру

    def forward(self, x):
        return self.head(self.norm(self.blocks(self.stem(x))))

def n_params(model):
    """Сколько обучаемых чисел в модели."""
    return sum(p.numel() for p in model.parameters())   # numel — число элементов тензора

def evaluate(model):
    """Точность на валидации."""
    model.eval()                                      # выключаем dropout
    with torch.no_grad():                             # градиенты не нужны — граф не строим
        # argmax — номер класса с наибольшим логитом; .cpu() — обратно на процессор, где лежат метки
        pred = model(X_val.to(device)).argmax(dim=1).cpu()
    return (pred == y_val).float().mean().item()      # доля верных ответов

print("параметров в DeepMLP:", n_params(DeepMLP()))

**Что получилось.** `DeepMLP` — около 400 тысяч параметров. В Colab с GPU первая строка покажет `cuda`.

Свой датасет `SlowDigits`: те же цифры, но каждый пример отдаётся с задержкой 2 мс — как чтение и декодирование файла. `train_epoch_timed` проводит эпоху и отдельно считает время **ожидания** батчей и время **вычислений**.

In [ ]:
class SlowDigits(Dataset):
    """Цифры, у которых чтение одного примера занимает 2 мс — как чтение и декодирование файла."""
    def __init__(self, X, y, delay=0.002):
        self.X, self.y, self.delay = X, y, delay

    def __len__(self):                                # DataLoader узнаёт отсюда, сколько всего примеров
        return len(self.X)

    def __getitem__(self, i):                         # DataLoader вызывает его для каждого номера в батче
        time.sleep(self.delay)                        # имитируем чтение файла с диска и декодирование
        return self.X[i], self.y[i]

def train_epoch_timed(model, loader, optimizer):
    """Одна эпоха обучения; отдельно считает время ожидания батчей и время вычислений."""
    loss_fn = nn.CrossEntropyLoss()
    t_data = t_compute = 0.0                          # два «секундомера»
    model.train()
    t0 = time.perf_counter()                          # t0 — момент, когда начали ждать батч
    for xb, yb in loader:                             # здесь DataLoader собирает очередной батч
        t1 = time.perf_counter()                      # t1 — батч получен
        t_data += t1 - t0                             # сколько ждали, пока DataLoader выдаст батч
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        if device == "cuda":
            torch.cuda.synchronize()                  # GPU работает асинхронно: ждём конца вычислений
        t0 = time.perf_counter()                      # шаг закончен — снова начинаем ждать батч
        t_compute += t0 - t1                          # сколько заняли перенос, проходы и шаг
    return t_data, t_compute

torch.manual_seed(0)
model = DeepMLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
slow = SlowDigits(X_train, y_train)
loader = DataLoader(slow, batch_size=64, shuffle=True)   # num_workers=0: батчи готовит сам цикл обучения
t_data, t_compute = train_epoch_timed(model, loader, optimizer)
print(f"ожидание данных {t_data:.2f} с | вычисления {t_compute:.2f} с")

**Что получилось.** Ожидание данных — больше 3 секунд за эпоху, вычисления — около 0.5 секунды. На каждые ~20 мс вычислений модель около 150 мс ждёт данных. Ускорять здесь вычисления бессмысленно.

### 1.3. DataLoader: готовим батчи параллельно

Тот же медленный датасет `slow`, но `DataLoader` с 0, 2 и 4 рабочими процессами (`num_workers`). Для каждого варианта — две эпохи, чтобы было видно цену первого запуска процессов.

**Процессы и потоки.** Процесс — запущенная программа со своей памятью; процессы работают одновременно на разных ядрах. Потоки внутри процесса делят его память, но из-за **GIL** Python-код в процессе выполняет только один поток. Поэтому `DataLoader` запускает именно процессы.

In [ ]:
for workers in [0, 2, 4]:                             # 0 — без рабочих процессов, как раньше
    # persistent_workers=True: рабочие процессы не завершаются между эпохами
    loader = DataLoader(slow, batch_size=64, shuffle=True, num_workers=workers,
                        persistent_workers=workers > 0)
    times = [train_epoch_timed(model, loader, optimizer) for _ in range(2)]   # две эпохи
    (d1, c1), (d2, c2) = times                        # (ожидание, вычисления) для эпох 1 и 2
    print(f"num_workers={workers}: эпоха 1 — ожидание {d1:.2f} с, эпоха 2 — ожидание {d2:.2f} с, вычисления {c2:.2f} с")

* `num_workers=2` — ожидание примерно вдвое меньше, `num_workers=4` — ещё вдвое.
* Первая эпоха с рабочими процессами чуть медленнее: процессы нужно запустить. `persistent_workers=True` оставляет их жить между эпохами.

В Colab всего 2 ядра процессора, но `time.sleep` их не нагружает, поэтому 4 процесса всё равно помогают. С настоящим декодированием выигрыш упрётся в число ядер.

Закреплённая память и асинхронный перенос — нужна видеокарта, поэтому код показан как текст.

```python
# фрагмент: перенос батчей на видеокарту без ожидания
loader = DataLoader(dataset, batch_size=256, shuffle=True,
                    num_workers=4, persistent_workers=True,
                    pin_memory=True)                  # батчи в закреплённой памяти
for xb, yb in loader:
    xb = xb.to("cuda", non_blocking=True)             # копирование идёт в фоне
    yb = yb.to("cuda", non_blocking=True)
    ...                                               # видеокарта начнёт считать, когда копия будет готова
```

### 1.4. Профилировщик: где тратятся вычисления

Профилировщик `torch.profiler` записывает каждую операцию за 10 шагов обучения. Сортируем по **собственному** времени (без вложенных вызовов) и печатаем шесть самых дорогих операций.

In [ ]:
from torch.profiler import profile, ProfilerActivity

# быстрый датасет без задержки: теперь нас интересуют только вычисления
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
# что записывать: операции на процессоре, а если есть видеокарта — и на ней
activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device == "cuda" else [])
with profile(activities=activities) as prof:          # записываем всё, что происходит внутри
    for i, (xb, yb) in enumerate(loader):
        if i == 10:                                   # 10 шагов обучения достаточно
            break
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()

# key_averages() — сводка: для каждой операции суммарное время и число вызовов за все 10 шагов
# сортируем по «собственному» времени (без вложенных вызовов); время — в микросекундах
events = sorted(prof.key_averages(), key=lambda e: e.self_cpu_time_total, reverse=True)
total = sum(e.self_cpu_time_total for e in events)
for e in events[:6]:                                  # шесть самых дорогих операций
    print(f"{e.key:32s} {e.self_cpu_time_total / 1000:6.1f} мс  {100 * e.self_cpu_time_total / total:4.1f}%  вызовов {e.count}")

* Время распределено между множеством мелких операций: LayerNorm, GELU, шаг AdamW.
* `sqrt` вызвана 780 раз за 10 шагов — по разу на каждый из 78 тензоров весов: Adam делит шаг на $\sqrt{v}$.

Много мелких вызовов — время уходит на накладные расходы. В больших сетях доминируют умножения матриц.

## Глава 2. Память и точность чисел

### 2.1. Что занимает память при обучении

Сколько памяти занимают веса, градиенты, состояния Adam и активации. `saved_bytes` перехватывает **хуком** каждый тензор, который autograd сохраняет для обратного прохода, и суммирует их размеры.

In [ ]:
def saved_bytes(model, x):
    """Сколько байт активаций autograd сохраняет для обратного прохода (без самих весов)."""
    # адреса весов в памяти: autograd сохраняет и веса, но их мы считаем отдельно
    param_ptrs = {p.data_ptr() for p in model.parameters()}
    seen, total = set(), 0

    def pack(t):                                      # вызывается для каждого сохраняемого тензора
        nonlocal total
        # один и тот же тензор могут сохранить несколько операций — узнаём его по адресу, размеру и типу
        key = (t.data_ptr(), t.numel(), t.dtype)
        if t.data_ptr() not in param_ptrs and key not in seen:   # один тензор считаем один раз
            seen.add(key)
            total += t.numel() * t.element_size()     # число элементов × байт на элемент
        return t                                      # сам тензор сохраняем как есть

    # pack — при сохранении тензора, вторая функция — при его использовании в обратном проходе
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        model(x).sum()                                # прямой проход: autograd сохраняет активации
    return total

MB = 2 ** 20                                          # мегабайт = 1 048 576 байт
x_big = torch.randn(256, 64)                          # батч из 256 примеров
for width, n_blocks in [(64, 12), (512, 24)]:         # наша сеть и её увеличенная версия
    torch.manual_seed(0)
    m = DeepMLP(width=width, n_blocks=n_blocks)
    p = n_params(m)
    act = saved_bytes(m, x_big)
    # float32: 4 байта на число; у Adam два буфера на каждый параметр (m и v)
    print(f"ширина {width}, блоков {n_blocks}: параметров {p / 1e6:5.2f} млн | веса {4 * p / MB:6.1f} МБ | "
          f"градиенты {4 * p / MB:6.1f} МБ | Adam {8 * p / MB:6.1f} МБ | активации {act / MB:6.1f} МБ")

* Веса, градиенты и Adam — $4 + 4 + 8 = 16$ байт на параметр.
* У маленькой сети активации батча из 256 примеров занимают больше, чем веса, градиенты и Adam вместе.
* Модель на 7 млрд параметров потребовала бы $7 \cdot 10^9 \cdot 16 = 112$ ГБ — больше, чем у одной видеокарты.

### 2.2. Форматы чисел: float32, float16, bfloat16

Свойства трёх форматов (`torch.finfo`) и что остаётся от чисел 70 000, $10^{-8}$ и 1.001 после перевода в 16 бит.

In [ ]:
for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    fi = torch.finfo(dtype)                           # свойства формата
    # bits — число бит; max — наибольшее число; tiny — наименьшее положительное без потери точности;
    # eps — расстояние от 1 до следующего представимого числа
    print(f"{str(dtype):15s} {fi.bits // 8} байта | максимум {fi.max:.2e} | "
          f"минимум {fi.tiny:.2e} | точность около 1: {fi.eps:.1e}")

x = torch.tensor([70000.0, 1e-8, 1.001])              # большое, очень маленькое и «1 с хвостиком»
for dtype in [torch.float16, torch.bfloat16]:
    # переводим в 16 бит и обратно в float32: видно, что осталось от числа после округления
    print(f"{str(dtype):15s}", x.to(dtype).float().tolist())

* **70 000** во float16 — `inf` (максимум 65 504), в bfloat16 — 70 144: шаг сетки около 70 000 равен 512.
* **$10^{-8}$** во float16 — ноль, в bfloat16 сохранилось.
* **1.001**: float16 сохранил почти точно, bfloat16 округлил до 1 — шаг сетки около единицы почти 0.008.

float16 точнее, bfloat16 — с диапазоном `float32`.

### 2.3. Смешанная точность: autocast

Внутри `torch.autocast` выполним линейный слой, всю сеть и потерю и посмотрим, в каком формате получился каждый результат.

In [ ]:
torch.manual_seed(0)
model = DeepMLP().to(device)
xb, yb = X_train[:64].to(device), y_train[:64].to(device)
# на GPU — float16 (или bfloat16 на новых картах), на процессоре — bfloat16
amp_dtype = torch.float16 if device == "cuda" else torch.bfloat16
with torch.autocast(device_type=device, dtype=amp_dtype):   # внутри блока формат выбирает autocast
    h = model.stem(xb)                                # линейный слой
    out = model(xb)                                   # вся сеть
    loss = nn.CrossEntropyLoss()(out, yb)             # потеря
# .dtype — формат, в котором хранится тензор
print("веса stem:   ", model.stem.weight.dtype)
print("выход stem:  ", h.dtype)
print("выход сети:  ", out.dtype)
print("потеря:      ", loss.dtype)
loss.backward()                                       # обратный проход — уже вне autocast
print("градиент stem:", model.stem.weight.grad.dtype)

* Веса и градиенты — `float32`: autocast их не меняет.
* Выход линейного слоя и сети — 16 бит (на процессоре — `bfloat16`, на GPU — `float16`).
* Потеря — `float32`: кросс-энтропия всегда считается в полной точности.

Обучим одну и ту же сеть без смешанной точности и с ней, замерим точность и время.

In [ ]:
def fit_amp(model, use_amp, epochs=30, lr=1e-3):
    """Обучает модель; use_amp=True — со смешанной точностью. Возвращает точность и время."""
    torch.manual_seed(0)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    # GradScaler нужен только для float16; при enabled=False он ничего не делает
    scaler = torch.amp.GradScaler(device, enabled=use_amp and amp_dtype == torch.float16)
    start = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            # при enabled=False блок with ничего не меняет — получается обычное обучение в float32
            with torch.autocast(device_type=device, dtype=amp_dtype, enabled=use_amp):
                loss = loss_fn(model(xb), yb)         # прямой проход и потеря — в смешанной точности
            # обратный проход и шаг — вне autocast
            scaler.scale(loss).backward()             # при float16: потеря умножена на масштаб
            scaler.step(optimizer)                    # градиенты делятся обратно; шаг пропускается, если есть inf
            scaler.update()                           # подстраиваем масштаб
    return evaluate(model), time.perf_counter() - start

for use_amp in [False, True]:                         # одна и та же сеть: без AMP и с AMP
    torch.manual_seed(0)
    acc, sec = fit_amp(DeepMLP().to(device), use_amp)
    print(f"AMP {'вкл ' if use_amp else 'выкл'}: точность на валидации {acc:.3f}, время {sec:.1f} с")

**Что получилось.** Точность почти одинаковая. На процессоре со смешанной точностью обучение **медленнее**: у него нет тензорных ядер. На видеокарте тот же код обычно ускоряет обучение в 2–3 раза.

### 2.4. GradScaler: защита маленьких градиентов в float16

Масштабирование потери: маленький градиент $10^{-8}$ во float16 без масштаба и с масштабом $2^{16}$.

In [ ]:
g = torch.tensor(1e-8)                                # маленький градиент
print("float16 без масштаба:", g.half().item())       # .half() — перевод в float16
scale = 65536.0                                       # 2^16 — типичный начальный масштаб GradScaler
g_scaled = (g * scale).half()                         # умножили, потом перевели в float16
# делим обратно уже в float32, где маленькие числа представимы
print("float16 с масштабом: ", g_scaled.item(), "→ после деления:", (g_scaled.float() / scale).item())

**Что получилось.** Без масштаба $10^{-8}$ обнулился. Умноженный на 65 536, он стал около $6.6 \cdot 10^{-4}$ — это float16 хранит, и после деления в `float32` получилось почти исходное значение. Так работает GradScaler.

### 2.5. Gradient checkpointing: пересчитать вместо того, чтобы хранить

Gradient checkpointing: те же блоки, но вызываются через `checkpoint(block, h)` — сохраняется только вход блока. Сравним память под активации и время прямого и обратного прохода на увеличенной сети.

In [ ]:
from torch.utils.checkpoint import checkpoint

class CheckpointedMLP(DeepMLP):
    """Та же сеть, но активации внутри блоков не хранятся, а пересчитываются при обратном проходе."""
    def forward(self, x):                             # веса и __init__ — от DeepMLP, меняем только forward
        h = self.stem(x)
        for block in self.blocks:
            # checkpoint(block, h) считает block(h), но запоминает только вход h;
            # при обратном проходе блок выполнится ещё раз, чтобы восстановить активации
            h = checkpoint(block, h, use_reentrant=False)
        return self.head(self.norm(h))

def fwd_bwd_time(model, x, repeats=3):
    """Среднее время прямого и обратного прохода."""
    start = time.perf_counter()
    for _ in range(repeats):
        model.zero_grad()
        model(x).sum().backward()                     # сумма выходов — простая «потеря» для обратного прохода
    return (time.perf_counter() - start) / repeats

for Model in [DeepMLP, CheckpointedMLP]:
    torch.manual_seed(0)                              # одинаковые веса у обеих сетей
    m = Model(width=512, n_blocks=24, dropout=0.0)    # увеличенная сеть из раздела 2.1
    print(f"{Model.__name__:16s}: активации {saved_bytes(m, x_big) / MB:6.1f} МБ, "
          f"прямой + обратный проход {fwd_bwd_time(m, x_big):.2f} с")

**Что получилось.** Активаций в 9 раз меньше: хранятся только входы 24 блоков. Цена — повторный прямой проход каждого блока при обратном.

## Глава 3. Большие батчи

### 3.1. Что меняет размер батча

Насколько градиент по случайному батчу отличается от градиента по всей обучающей выборке: относительная ошибка $\lVert g_B - g \rVert / \lVert g \rVert$ для батчей 16, 64 и 256. Здесь же определяется `loss_fn` — она нужна дальше.

In [ ]:
torch.manual_seed(0)
model = DeepMLP(dropout=0.0)                          # без dropout: случайность только в выборе примеров
loss_fn = nn.CrossEntropyLoss()

def grad_vector(xb, yb):
    """Градиент потери по всем параметрам, вытянутый в один вектор."""
    model.zero_grad()
    loss_fn(model(xb), yb).backward()
    # flatten вытягивает градиент каждого тензора весов в строку, cat склеивает их в один длинный вектор
    return torch.cat([p.grad.flatten() for p in model.parameters()])

g_full = grad_vector(X_train, y_train)                # «настоящий» градиент — по всем 1347 примерам
for B in [16, 64, 256]:
    errors = []
    for _ in range(50):                               # 50 случайных батчей размера B
        idx = torch.randperm(len(X_train))[:B]        # B случайных номеров примеров без повторов
        g = grad_vector(X_train[idx], y_train[idx])
        # относительная ошибка: длина разницы, делённая на длину настоящего градиента
        errors.append(((g - g_full).norm() / g_full.norm()).item())
    print(f"батч {B:3d}: шагов за эпоху {len(X_train) // B:3d} | относительная ошибка градиента {sum(errors) / len(errors):.2f}")

**Что получилось.** Ошибка: 1.45 для батча 16, 0.72 для 64, 0.34 для 256. При увеличении батча в 4 раза ошибка падает в 2 раза — как $1/\sqrt{B}$. Зато шагов за эпоху в 4 раза меньше.

### 3.2. Скорость обучения и разогрев

Расписание скорости обучения: линейный разогрев 100 шагов, потом снижение по косинусу. Обучать модель не нужно — только записываем скорость обучения на каждом шаге.

In [ ]:
model = DeepMLP()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)   # lr=1e-3 — целевая скорость обучения
total_steps, warmup_steps = 1000, 100
# разогрев: от 1% до 100% за warmup_steps шагов, потом плавное снижение по косинусу до нуля
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps)
# SequentialLR выполняет расписания по очереди: на шаге milestones[0] переключается с warmup на cosine
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_steps])

lrs = []
for step in range(total_steps):
    lrs.append(scheduler.get_last_lr()[0])            # скорость обучения на этом шаге ([0] — первая группа весов)
    optimizer.step()                                  # здесь был бы обычный шаг обучения
    scheduler.step()                                  # расписание сдвигается после каждого шага
print("скорость обучения:", {s: f"{lrs[s]:.1e}" for s in [0, 50, 100, 300, 550, 800, 999]})

**Что получилось.** За первые 100 шагов скорость обучения выросла от $10^{-5}$ до $10^{-3}$, к шагу 550 упала вдвое, к концу — почти до нуля.

### 3.3. Накопление градиентов

Накопление градиентов: 4 микробатча по 16 примеров с делением потери на 4 должны дать тот же градиент, что один батч из 64. И что будет, если забыть разделить.

In [ ]:
torch.manual_seed(0)
model = DeepMLP(dropout=0.0)                          # без dropout, чтобы проходы были детерминированы
xb, yb = X_train[:64], y_train[:64]

model.zero_grad()
loss_fn(model(xb), yb).backward()                     # один большой батч из 64 примеров
g_big = [p.grad.clone() for p in model.parameters()]  # clone — копия: .grad дальше перезапишется

accum_steps = 4
model.zero_grad()                                     # обнуляем один раз — перед всеми микробатчами
for xm, ym in zip(xb.chunk(accum_steps), yb.chunk(accum_steps)):   # chunk режет батч на 4 микробатча по 16
    (loss_fn(model(xm), ym) / accum_steps).backward()              # градиенты складываются в .grad
# наибольшее по модулю отличие среди всех весов
diff = max((p.grad - g).abs().max().item() for p, g in zip(model.parameters(), g_big))
print(f"наибольшее расхождение с большим батчем: {diff:.1e}")

model.zero_grad()
for xm, ym in zip(xb.chunk(accum_steps), yb.chunk(accum_steps)):
    loss_fn(model(xm), ym).backward()                 # забыли разделить на accum_steps
# во сколько раз длина накопленного градиента больше длины градиента большого батча
ratio = (torch.cat([p.grad.flatten() for p in model.parameters()]).norm()
         / torch.cat([g.flatten() for g in g_big]).norm())
print(f"без деления на accum_steps градиент больше в {ratio:.2f} раза")

**Что получилось.** Расхождение $10^{-7}$ — ошибки округления: градиенты совпали. Без деления на `accum_steps` градиент ровно в 4 раза больше — как незаметное увеличение скорости обучения в 4 раза.

Так выглядит цикл обучения с накоплением:

```python
# фрагмент: цикл обучения с накоплением градиентов
accum_steps = 4                                       # эффективный батч = batch_size · accum_steps
optimizer.zero_grad()
for i, (xb, yb) in enumerate(loader):
    loss = loss_fn(model(xb), yb) / accum_steps       # делим, чтобы получить среднее по большому батчу
    loss.backward()                                   # градиенты складываются в .grad
    if (i + 1) % accum_steps == 0:                    # каждые accum_steps микробатчей —
        optimizer.step()                              # один шаг оптимизатора
        optimizer.zero_grad()
```

### 3.4. Ограничение нормы градиента

Ограничение нормы градиента с маленьким порогом 0.5 — чтобы обрезка точно сработала.

In [ ]:
model.zero_grad()
loss_fn(model(xb), yb).backward()                     # обычный градиент по батчу из 64 примеров
# подчёркивание в конце имени: функция меняет .grad на месте; возвращает норму ДО обрезки
norm_before = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
norm_after = torch.cat([p.grad.flatten() for p in model.parameters()]).norm()   # длина после обрезки
print(f"норма градиента до: {norm_before:.3f}, после: {norm_after:.3f}")

**Что получилось.** Норма градиента была около 4.6, после ограничения — ровно 0.5. Функция возвращает норму **до** ограничения — её полезно записывать в логи.

В Lightning всё это включается параметрами `Trainer`:

```python
# фрагмент: смешанная точность, накопление и ограничение градиента в Lightning
trainer = L.Trainer(
    precision="bf16-mixed",                           # или "16-mixed" — float16 с GradScaler
    accumulate_grad_batches=4,                        # накопление градиентов
    gradient_clip_val=1.0,                            # ограничение нормы градиента
    max_epochs=30,
)
```

## Глава 4. Обучение на нескольких устройствах

### 4.1. Параллелизм по данным: идея

Параллелизм по данным на двух копиях модели: среднее градиентов по половинам батча против градиента по всему батчу.

In [ ]:
import copy

torch.manual_seed(0)
model = DeepMLP(dropout=0.0)
replicas = [copy.deepcopy(model) for _ in range(2)]   # «два GPU»: независимые копии с одинаковыми весами
xb, yb = X_train[:64], y_train[:64]

model.zero_grad()
loss_fn(model(xb), yb).backward()                     # эталон: весь батч на одном устройстве

for replica, xh, yh in zip(replicas, xb.chunk(2), yb.chunk(2)):   # chunk(2) — две половины по 32 примера
    replica.zero_grad()
    loss_fn(replica(xh), yh).backward()               # каждая копия — на своей половине батча

# AllReduce: усредняем градиенты копий
diff = 0.0
# идём по весам параллельно: p — эталонной модели, p0 и p1 — тот же вес в двух копиях
for p, p0, p1 in zip(model.parameters(), *[r.parameters() for r in replicas]):
    avg = (p0.grad + p1.grad) / 2
    diff = max(diff, (avg - p.grad).abs().max().item())
print(f"среднее градиентов двух копий vs градиент всего батча: расхождение {diff:.1e}")

**Что получилось.** Среднее градиентов двух копий совпало с градиентом всего батча до ошибок округления. Одинаковые градиенты и одинаковый шаг — веса копий остаются одинаковыми.

### 4.3. DDP на практике

Настоящее обучение с DDP на двух процессах. Видеокарты не нужны: на процессоре работает библиотека Gloo. Ячейка записывает скрипт `ddp_demo.py` и запускает его через `torchrun` (`python -m torch.distributed.run`). Выполняется около минуты.

In [ ]:
import subprocess
import sys

# текст скрипта, который выполнит каждый из двух процессов
script = r'''
import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

# подключаемся к остальным процессам; адрес для связи и номера процессов
# torchrun передал через переменные окружения
dist.init_process_group(backend="gloo")         # на видеокартах NVIDIA — backend="nccl"
rank, world_size = dist.get_rank(), dist.get_world_size()   # номер этого процесса и сколько их всего

X, y = load_digits(return_X_y=True)
X = torch.tensor(StandardScaler().fit_transform(X), dtype=torch.float32)
dataset = TensorDataset(X, torch.tensor(y))
# каждый процесс получает свою половину примеров
sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=42)
loader = DataLoader(dataset, batch_size=32, sampler=sampler)   # shuffle не указываем: порядок задаёт sampler

torch.manual_seed(0)
# обёртка DDP рассылает веса процесса 0 остальным и усредняет градиенты при backward()
model = DDP(nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10)))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
first_ids = []
for epoch in range(5):
    sampler.set_epoch(epoch)                    # новое перемешивание в каждой эпохе
    first_ids.append(next(iter(sampler)))       # номер первого примера эпохи — для проверки
    total, count = 0.0, 0
    for xb, yb in loader:                       # обычный цикл обучения, как на одном устройстве
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()                         # здесь DDP усредняет градиенты между процессами
        optimizer.step()
        total, count = total + loss.item() * len(xb), count + len(xb)

# собираем сведения со всех процессов в процесс 0
info = {"примеров": len(sampler), "батчей": len(loader), "первые примеры эпох": first_ids,
        "средняя потеря за эпоху": round(total / count, 4),
        "сумма весов": round(sum(p.sum() for p in model.parameters()).item(), 4)}
gathered = [None] * world_size
dist.all_gather_object(gathered, info)          # каждый процесс получает словари всех процессов
if rank == 0:                                   # печатает только процесс 0, иначе вывод задвоится
    for r, inf in enumerate(gathered):
        print(f"процесс {r}:", inf)
dist.destroy_process_group()                    # отключаемся от остальных процессов
'''
open("ddp_demo.py", "w").write(script)
# torchrun запускает 2 процесса и задаёт им RANK, WORLD_SIZE и адрес для связи
result = subprocess.run([sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2", "ddp_demo.py"],
                        capture_output=True, text=True)
# при успехе печатаем вывод процессов, при ошибке — конец сообщения об ошибке
print(result.stdout if result.returncode == 0 else result.stderr[-2000:])

* **Данные разделены**: у каждого процесса 899 примеров — половина из 1797 (одна дополнена повтором).
* **Перемешивание разное** в каждой эпохе — благодаря `sampler.set_epoch(epoch)`.
* **Веса одинаковые**: сумма весов совпала до последнего знака, хотя процессы видели разные данные.

На видеокартах запуск выглядит так:

```bash
# одна машина, 4 видеокарты
torchrun --nproc_per_node=4 train.py

# две машины по 4 видеокарты; на машине 0 (главной):
torchrun --nproc_per_node=4 --nnodes=2 --node_rank=0 --master_addr=192.168.1.1 --master_port=29500 train.py
# на машине 1 — то же самое, но --node_rank=1
```

DDP в Lightning:

```python
# фрагмент: DDP в Lightning на 4 видеокартах
trainer = L.Trainer(accelerator="gpu", devices=4, strategy="ddp", max_epochs=30)
trainer.fit(model, datamodule=dm)
```

### 4.4. Когда модель не помещается: параллелизм модели

Тензорный параллелизм: матрица весов делится между двумя «видеокартами» по столбцам или по строкам.

In [ ]:
torch.manual_seed(0)
X_ = torch.randn(4, 6)                                # 4 примера, 6 признаков
W = torch.randn(6, 8)                                 # веса слоя: 6 входов → 8 выходов

# по столбцам: «GPU 0» хранит выходы 0–3, «GPU 1» — выходы 4–7; результаты склеиваем
# W[:, :4] — первые 4 столбца (6×4), каждое произведение даёт матрицу 4×4
Y_cols = torch.cat([X_ @ W[:, :4], X_ @ W[:, 4:]], dim=1)
# по строкам: каждый «GPU» берёт свою половину входов; результаты складываем (AllReduce)
# X_[:, :3] — первые 3 признака, W[:3] — первые 3 строки весов; каждое произведение — уже полный выход 4×8
Y_rows = X_[:, :3] @ W[:3] + X_[:, 3:] @ W[3:]
# allclose — равны ли матрицы с точностью до ошибок округления
print("деление по столбцам совпало:", torch.allclose(X_ @ W, Y_cols))
print("деление по строкам совпало: ", torch.allclose(X_ @ W, Y_rows, atol=1e-6))

**Что получилось.** Оба способа дают тот же результат, что умножение на всю матрицу.

### 4.5. ZeRO и FSDP: не хранить лишние копии

Память на одну видеокарту для модели на 7.5 млрд параметров и 64 видеокарт: DDP и три стадии ZeRO.

In [ ]:
def zero_memory_gb(P, N):
    """Память на одну видеокарту (ГБ) при обучении со смешанной точностью и Adam.

    P — число параметров, N — число видеокарт. На параметр: 2 байта весов float16,
    2 байта градиентов float16 и 12 байт состояний оптимизатора (копия весов float32, m и v).
    """
    w, g, s = 2 * P, 2 * P, 12 * P                    # байт: веса, градиенты, состояния оптимизатора
    return {"DDP": (w + g + s) / 1e9,                 # каждая видеокарта хранит всё
            "ZeRO-1": (w + g + s / N) / 1e9,           # делим состояния оптимизатора
            "ZeRO-2": (w + (g + s) / N) / 1e9,         # + градиенты
            "ZeRO-3": ((w + g + s) / N) / 1e9}         # + сами веса

for name, gb in zero_memory_gb(P=7.5e9, N=64).items():   # модель на 7.5 млрд параметров, 64 GPU
    print(f"{name:7s}: {gb:6.1f} ГБ на видеокарту")

**Что получилось.** DDP — 120 ГБ на видеокарту, ZeRO-1 — 31.4, ZeRO-2 — 16.6, ZeRO-3 — 1.9 ГБ, в 64 раза меньше. Состояния оптимизатора — три четверти памяти, поэтому главный выигрыш даёт уже ZeRO-1.

FSDP и DeepSpeed в Lightning:

```python
# фрагмент: FSDP и DeepSpeed в Lightning
trainer = L.Trainer(accelerator="gpu", devices=8, strategy="fsdp")            # PyTorch FSDP
trainer = L.Trainer(accelerator="gpu", devices=8, strategy="deepspeed_stage_2")  # DeepSpeed ZeRO-2
```

## Глава 5. Цикл обучения без лишних потерь

### 5.1. Лишние синхронизации

Глава 5 — приёмы для видеокарты, поэтому код показан как текст. Копим потерю на видеокарте, `.item()` — один раз за эпоху:

```python
# фрагмент: копим потерю на видеокарте, .item() — один раз за эпоху
total = torch.zeros((), device="cuda")                # ноль на видеокарте
for xb, yb in loader:
    ...                                               # обычный шаг обучения
    total += loss.detach()                            # сложение на видеокарте, без ожидания
print("средняя потеря:", (total / len(loader)).item())   # одна синхронизация за эпоху
```

### 5.2. torch.compile

Компиляция модели:

```python
# фрагмент: компиляция модели
model = DeepMLP().to("cuda")
model = torch.compile(model)                          # первый шаг — долгий: идёт компиляция
# дальше модель используется как обычно; последующие шаги быстрее
```

### 5.3. Воспроизводимость и скорость

Две крайние настройки — воспроизводимость и скорость:

```python
# фрагмент: две крайние настройки
# максимальная воспроизводимость (медленнее)
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)              # только детерминированные алгоритмы
torch.backends.cudnn.benchmark = False

# максимальная скорость (результаты могут немного отличаться между запусками)
torch.backends.cudnn.benchmark = True                 # подбирать самый быстрый алгоритм свёртки
```

### 5.4. Чекпоинты: сохранить и продолжить

Чекпоинт: обучаем 3 эпохи, сохраняем и продолжаем тремя способами — без перерыва (A), из полного чекпоинта (B) и загрузив только веса модели (C).

In [ ]:
def make_run():
    """Новая модель, оптимизатор и расписание — как в начале обучения."""
    torch.manual_seed(0)
    model = DeepMLP()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)   # расписание на 10 эпох
    return model, optimizer, scheduler

def train_epoch(model, optimizer, scheduler, epoch):
    """Одна эпоха; порядок батчей зависит только от номера эпохи."""
    # свой генератор с сидом = номер эпохи: порядок батчей в эпохе одинаков во всех трёх вариантах ниже
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                        generator=torch.Generator().manual_seed(epoch))
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()                                  # расписание здесь шагает раз в эпоху
    return loss.item()                                # потеря на последнем батче эпохи

model, optimizer, scheduler = make_run()
for epoch in range(3):                                # обучаем эпохи 0, 1, 2
    train_epoch(model, optimizer, scheduler, epoch)
# state_dict() — словарь со всеми числами объекта; torch.save записывает словарь в файл
torch.save({"epoch": 3,                               # всё, что нужно для продолжения
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),      # моменты Adam
            "scheduler": scheduler.state_dict(),      # где мы в расписании
            "rng": torch.get_rng_state()},            # состояние генератора случайных чисел (dropout)
           "checkpoint.pt")
loss_a = train_epoch(model, optimizer, scheduler, 3)  # вариант A: без перерыва — эпоха 4

model, optimizer, scheduler = make_run()              # вариант B: «перезапуск» — всё с нуля...
ckpt = torch.load("checkpoint.pt")
model.load_state_dict(ckpt["model"])                  # ...и восстанавливаем каждую часть из чекпоинта
optimizer.load_state_dict(ckpt["optimizer"])
scheduler.load_state_dict(ckpt["scheduler"])
torch.set_rng_state(ckpt["rng"])
loss_b = train_epoch(model, optimizer, scheduler, ckpt["epoch"])

model, optimizer, scheduler = make_run()              # вариант C: а если загрузить только веса модели?
model.load_state_dict(ckpt["model"])
torch.set_rng_state(ckpt["rng"])
loss_c = train_epoch(model, optimizer, scheduler, ckpt["epoch"])
print(f"без перерыва {loss_a:.6f} | из полного чекпоинта {loss_b:.6f} | только веса {loss_c:.6f}")

* **Из полного чекпоинта** обучение продолжилось точно так же, как без перерыва: потеря совпала до шестого знака.
* **Только веса** — потеря другая: оптимизатор начал с нулевых $m$ и $v$, а расписание — с начала.

## Глава 6. Аугментации для табличных данных

### 6.2. Mixup: смешиваем примеры

Две аугментации для табличных данных: гауссов шум и mixup — смесь двух примеров вместе с метками.

In [ ]:
def add_noise(x, std=0.1):
    """Небольшой гауссов шум к каждому признаку."""
    return x + torch.randn_like(x) * std              # randn_like — случайные числа той же формы, что x

def mixup(x, y, n_classes=10, alpha=0.4):
    """Смешивает каждый пример со случайным другим; метки становятся «мягкими»."""
    lam = torch.distributions.Beta(alpha, alpha).sample().item()   # доля первого примера, одна на весь батч
    idx = torch.randperm(len(x))                      # с кем смешиваем: x[idx] — те же примеры в другом порядке
    y_soft = nn.functional.one_hot(y, n_classes).float()   # номер класса → вектор из 10 чисел с одной единицей
    return lam * x + (1 - lam) * x[idx], lam * y_soft + (1 - lam) * y_soft[idx]

torch.manual_seed(1)
x_mix, y_mix = mixup(X_train[:8], y_train[:8])        # смешиваем первые 8 примеров
print("метки исходных примеров:", y_train[:8].tolist())
print("мягкая метка примера 0: ", [round(v, 2) for v in y_mix[0].tolist()])
# CrossEntropyLoss принимает и номер класса, и вектор вероятностей
print("потеря с мягкими метками:", round(nn.CrossEntropyLoss()(DeepMLP()(x_mix), y_mix).item(), 3))

**Что получилось.** Пример 0 — двойка, смешанная с семёркой в пропорции 0.8 : 0.2; метка стала вектором с двумя ненулевыми числами. `CrossEntropyLoss` принимает такие мягкие метки вместо номера класса.

### 6.3. Проверим на маленькой выборке

Обучим `DeepMLP` всего на 200 примерах без аугментаций, с шумом и с mixup; каждый вариант — три запуска. Это самая долгая ячейка — около минуты на процессоре.

In [ ]:
def fit_small(augment, n_train=200, epochs=150, seed=0):
    """Обучает DeepMLP на n_train примерах с аугментацией augment и возвращает точность на валидации."""
    torch.manual_seed(seed)
    model = DeepMLP()
    # только первые n_train обучающих примеров; валидация — прежняя, все 450 примеров
    loader = DataLoader(TensorDataset(X_train[:n_train], y_train[:n_train]), batch_size=32, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            # аугментация применяется к каждому батчу заново — в каждой эпохе искажения новые
            if augment == "шум":
                xb = add_noise(xb)
            elif augment == "mixup":
                xb, yb = mixup(xb, yb)                # yb теперь мягкие метки
            optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(model(xb), yb)
            loss.backward()
            optimizer.step()
    return evaluate(model)                            # валидация — без аугментаций

for augment in ["нет", "шум", "mixup"]:
    accs = [fit_small(augment, seed=s) for s in range(3)]   # три запуска с разными начальными весами
    print(f"аугментация {augment:5s}: точность {sum(accs) / 3:.3f}  (запуски: {', '.join(f'{a:.3f}' for a in accs)})")

**Что получилось.** Mixup дал небольшой прирост средней точности, шум — не помог. Но разброс между запусками сравним с разницей между вариантами: на трёх запусках выводы предварительные.